# FaceRestore Training Pipeline - Kaggle T4 x2 Edition

Converted from Google Colab single-GPU to **Kaggle dual T4** setup.

## Kaggle Setup (do this FIRST)

1. **Settings -> Accelerator**: Select **T4 x2** (2 GPUs)
2. **Settings -> Internet**: Turn **ON** (needed for mask download)
3. **Add Data** (right panel -> Add Dataset):
   - **`celeba-256`** -- your preprocessed 256x256 face images (zip or folder)
   - **`facerestore-checkpoints`** -- upload **just `last.pth`** from your Drive folder (see note below)
   - **`inpaint-training`** -- upload `inpaint_training.zip` from your machine

### Which checkpoint files to upload?

**Only `last.pth` is needed.** It contains everything to resume training:
- Model weights (same as what's in `epoch_005.pth`)
- Epoch number (so it knows to start from epoch 6)
- Optimizer state (Adam momentum buffers)
- Scheduler state (learning rate decay progress)

The individual `epoch_001.pth` through `epoch_005.pth` files are **just model weights** -- useful if you want to compare or roll back, but they're NOT needed for resuming. Skip them to save upload time and dataset size.

## Logging & Metrics

This notebook logs to **two places simultaneously**:

| Logger | Where to View | What It Shows |
|--------|--------------|---------------|
| **Kaggle TensorBoard** | Output tab -> TensorBoard | Loss curves, learning rate, epoch progress |
| **W&B** (optional) | wandb.ai dashboard | Same + project comparison, run history |

Kaggle has **built-in TensorBoard support** -- no setup needed, just click the TensorBoard button in the Output tab after training starts. This is the easiest way to monitor your training.

## What Changed from Colab Version

| Feature | Colab | Kaggle (this notebook) |
|---------|-------|------------------------|
| GPUs | 1x T4 | 2x T4 (DataParallel) |
| Time limit | ~5 hrs/account | 30 hrs/week |
| Storage | Google Drive | `/kaggle/input/` + `/kaggle/working/` |
| Batch size | 16 | 32 (16 per GPU) |
| Workers | 2 | 4 |
| Epochs/session | ~1 | ~13-15 |
| Logging | W&B (1 run/epoch, fragmented) | Kaggle TensorBoard + W&B (1 continuous run) |
| Checkpoints | Drive (auto-persist) | `/kaggle/working/` (download between sessions) |

## Estimated Training Speed

- Single T4: ~3.5 hrs/epoch -> **~1.8-2.2 hrs/epoch** with 2x T4 DataParallel
- 30 hrs total -> **~13-15 epochs** per weekly quota
- You already have 5 epochs, so ~2-3 weekly sessions to reach 20+ epochs


In [ ]:
# Cell 1: Install Dependencies
import os

# Install packages (TensorBoard is pre-installed on Kaggle)
!pip install -q jsonlines tqdm "matplotlib<3.8" wandb tensorboard

# W&B Authentication (optional - Kaggle TensorBoard works without this)
try:
    import wandb
    wandb_key = os.environ.get('WANDB_API_KEY')
    if wandb_key:
        wandb.login(key=wandb_key)
        print('W&B: Authenticated via Kaggle secret')
    else:
        # Try interactive login (works if Internet is ON)
        wandb.login()
        print('W&B: Manual login (paste your key when prompted)')
except Exception as e:
    print(f'W&B: Skipped ({e})')
    print('Kaggle TensorBoard will still work for logging!')

print('\nDependencies installed.')
print('Logging: Kaggle TensorBoard (built-in) + W&B (if authenticated)')

In [ ]:
# Cell 2: Setup Project Structure & Prepare Data
import os, shutil

KAGGLE_INPUT = '/kaggle/input'
WORK_DIR     = '/kaggle/working'
PROJECT_DIR  = os.path.join(WORK_DIR, 'inpaint_project')
DATASET_DIR  = os.path.join(WORK_DIR, 'dataset')
CKPT_DIR     = os.path.join(WORK_DIR, 'facerestore_checkpoints')
TB_LOG_DIR   = os.path.join(WORK_DIR, 'logs')  # TensorBoard log directory

# --- Unzip training code ---
inpaint_zip = None
for d in os.listdir(KAGGLE_INPUT):
    candidate = os.path.join(KAGGLE_INPUT, d, 'inpaint_training.zip')
    if os.path.exists(candidate):
        inpaint_zip = candidate
        break

if inpaint_zip and not os.path.exists(PROJECT_DIR):
    !unzip -qo {inpaint_zip} -d {PROJECT_DIR}
    print(f'Unzipped training code to {PROJECT_DIR}')
elif not os.path.exists(PROJECT_DIR):
    for d in os.listdir(KAGGLE_INPUT):
        candidate = os.path.join(KAGGLE_INPUT, d)
        if os.path.isdir(candidate) and os.path.exists(os.path.join(candidate, 'inpaint', 'module.py')):
            shutil.copytree(candidate, PROJECT_DIR)
            print(f'Copied training code from {candidate}')
            break
    else:
        print('WARNING: inpaint_training.zip not found in /kaggle/input/')
        print('Available datasets:', os.listdir(KAGGLE_INPUT))

# --- Setup face images ---
faces_dir = os.path.join(DATASET_DIR, 'faces')
os.makedirs(faces_dir, exist_ok=True)

celeba_found = False
for d in os.listdir(KAGGLE_INPUT):
    candidate = os.path.join(KAGGLE_INPUT, d)
    if not os.path.isdir(candidate):
        continue
    if 'celeba' not in d.lower():
        continue
    # Check for zip
    zip_path = os.path.join(candidate, 'celeba_256.zip')
    if os.path.exists(zip_path):
        !unzip -qo {zip_path} -d {faces_dir}
        print(f'Unzipped face images from {zip_path}')
        celeba_found = True
        break
    # Check if images are directly in the folder or a subfolder
    image_dirs = [candidate]
    for sub in os.listdir(candidate):
        subpath = os.path.join(candidate, sub)
        if os.path.isdir(subpath):
            image_dirs.append(subpath)
    for img_dir in image_dirs:
        if any(f.lower().endswith(('.jpg', '.png', '.jpeg')) for f in os.listdir(img_dir)[:10]):
            count = 0
            for f in os.listdir(img_dir):
                if f.lower().endswith(('.jpg', '.png', '.jpeg')):
                    src = os.path.join(img_dir, f)
                    dst = os.path.join(faces_dir, f)
                    if not os.path.exists(dst):
                        os.link(src, dst)  # hard link (fast, no extra disk)
                        count += 1
            if count > 0:
                print(f'Linked {count} face images from {img_dir}')
                celeba_found = True
                break
    if celeba_found:
        break

if not celeba_found:
    print('WARNING: CelebA 256x256 images not found in /kaggle/input/')
    print('Available datasets:', os.listdir(KAGGLE_INPUT))

# --- Setup existing checkpoint (just last.pth is enough!) ---
os.makedirs(CKPT_DIR, exist_ok=True)
ckpt_found = False
for d in os.listdir(KAGGLE_INPUT):
    candidate = os.path.join(KAGGLE_INPUT, d)
    if not os.path.isdir(candidate):
        continue
    if 'checkpoint' in d.lower() or 'facerestore' in d.lower():
        for f in os.listdir(candidate):
            src = os.path.join(candidate, f)
            dst = os.path.join(CKPT_DIR, f)
            # Only copy last.pth -- it has everything needed to resume
            # epoch_00X.pth files are just weights, not needed for resuming
            if f == 'last.pth' and not os.path.exists(dst):
                shutil.copy2(src, dst)
                print(f'  Copied: {f} (has model + epoch + optimizer + scheduler)')
                ckpt_found = True
            elif f.endswith('.pth') and f != 'last.pth' and not os.path.exists(dst):
                # Optionally copy other .pth files too (not required for resuming)
                # Uncomment the next line if you want them:
                # shutil.copy2(src, dst)
                print(f'  Skipped: {f} (not needed for resume)')
if ckpt_found:
    print(f'\nCheckpoint ready in {CKPT_DIR}')
    print('last.pth contains: model weights + epoch number + optimizer + scheduler')
else:
    print('No last.pth found -- will train from base model.state_dict')

# --- Copy base model for fine-tuning ---
base_model_path = None
for d in os.listdir(KAGGLE_INPUT):
    candidate = os.path.join(KAGGLE_INPUT, d, 'model.state_dict')
    if os.path.exists(candidate):
        base_model_path = candidate
        break
    candidate2 = os.path.join(KAGGLE_INPUT, d)
    if os.path.isdir(candidate2):
        for f in os.listdir(candidate2):
            if f == 'model.state_dict':
                base_model_path = os.path.join(candidate2, f)
                break
    if base_model_path:
        break

if base_model_path:
    shutil.copy2(base_model_path, os.path.join(WORK_DIR, 'model.state_dict'))
    print(f'Base model copied from {base_model_path}')
else:
    print('WARNING: model.state_dict not found -- training from scratch!')

# --- Download QuickDraw mask data ---
masks_dir = os.path.join(DATASET_DIR, 'masks')
os.makedirs(masks_dir, exist_ok=True)
mask_file = os.path.join(masks_dir, 'face.ndjson')
if not os.path.exists(mask_file):
    !wget -q https://storage.googleapis.com/quickdraw_dataset/full/simplified/face.ndjson -P {masks_dir}
    print('Downloaded QuickDraw face.ndjson masks')
else:
    print('QuickDraw masks already present')

# --- Verification ---
print('\n' + '='*60)
print('SETUP VERIFICATION')
print('='*60)
face_count = len(os.listdir(faces_dir)) if os.path.exists(faces_dir) else 0
print(f'Face images:      {face_count}')
print(f'Mask files:       {os.listdir(masks_dir)}')
print(f'Base model:       {os.path.exists(os.path.join(WORK_DIR, "model.state_dict"))}')
print(f'Train.py:         {os.path.exists(os.path.join(PROJECT_DIR, "train.py"))}')
print(f'Checkpoint dir:   {CKPT_DIR} ({len(os.listdir(CKPT_DIR))} files)')
print(f'TensorBoard logs: {TB_LOG_DIR}')
print()
if face_count > 1000:
    print('ALL READY -- proceed to Cell 3!')
else:
    print('ERROR: Not enough face images. Check your Kaggle dataset setup.')

In [ ]:
# Cell 3: Verify Multi-GPU Setup
import torch

n_gpus = torch.cuda.device_count()
print(f'CUDA available:  {torch.cuda.is_available()}')
print(f'GPU count:       {n_gpus}')

if n_gpus > 0:
    for i in range(n_gpus):
        props = torch.cuda.get_device_properties(i)
        print(f'  GPU {i}: {props.name} ({props.total_mem / 1e9:.1f} GB)')
    
    if n_gpus >= 2:
        print(f'\nDataParallel will distribute batches across {n_gpus} GPUs')
        print(f'  Effective batch: 32 total (16 per GPU)')
    else:
        print(f'\nSingle GPU -- DataParallel NOT enabled')
        print(f'  Consider switching accelerator to T4 x2 for 2x speed')
else:
    print('\nWARNING: No GPU detected! Set Accelerator to T4 x2 in Settings.')

In [ ]:
# Cell 4: Write Modified train.py with DataParallel + TensorBoard + W&B
#
# Changes from the Colab version:
#   1. nn.DataParallel auto-wrapping for multi-GPU
#   2. All checkpoint saves use clean state_dict (no module. prefix)
#   3. Kaggle TensorBoard logging (built-in, no setup needed)
#   4. W&B logging (optional, for cloud dashboard)
#   5. Default workers increased to 4

import os

TRAIN_PY = r'''import argparse
from pathlib import Path

import torch
import torch.nn as nn
from tqdm import tqdm

# TensorBoard (Kaggle-native, always available)
from torch.utils.tensorboard import SummaryWriter

# W&B (optional cloud dashboard)
try:
    import wandb
    HAS_WANDB = True
except ImportError:
    HAS_WANDB = False

try:
    from .data import make_dataloader
    from .module import InpaintLoss, InpaintNet
except ImportError:
    from data import make_dataloader
    from module import InpaintLoss, InpaintNet


def expand_mask(mask_batch, n_channels):
    return mask_batch.unsqueeze(1).repeat(1, n_channels, 1, 1)


def preprocess_batch(batch, device, n_channels):
    image_batch, mask_batch = batch
    image_batch = image_batch.float().to(device)
    mask_batch = expand_mask(mask_batch.float().to(device), n_channels)
    image_masked_batch = image_batch * mask_batch
    return image_batch, image_masked_batch, mask_batch


def get_model_state_dict(model):
    """Get state dict without DataParallel module. prefix.
    Ensures checkpoints are portable between single-GPU and multi-GPU."""
    if isinstance(model, (nn.DataParallel, nn.parallel.DistributedDataParallel)):
        return model.module.state_dict()
    return model.state_dict()


def run_epoch(model, criterion, dataloader, device, optimizer=None, epoch_num=0):
    training = optimizer is not None
    raw_model = model.module if isinstance(model, nn.DataParallel) else model
    raw_model.train(training)
    total_loss = 0.0

    desc = f"Epoch {epoch_num} {'[train]' if training else '[val]'}"
    pbar = tqdm(dataloader, desc=desc, unit="batch", leave=True)

    for batch in pbar:
        image_batch, image_masked_batch, mask_batch = preprocess_batch(
            batch, device, raw_model.in_channels
        )

        if training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(training):
            output_batch, _ = model(image_masked_batch, mask_batch)
            loss = criterion(output_batch, mask_batch, image_batch).mean()

        if training:
            loss.backward()
            optimizer.step()

        total_loss += loss.item()
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    return total_loss / max(len(dataloader), 1)


def make_loader(path, masks_dir, batch_size, workers, val):
    return make_dataloader(
        path,
        masks_dir,
        val=val,
        batch_size=batch_size,
        shuffle=not val,
        drop_last=not val,
        num_workers=workers,
        pin_memory=torch.cuda.is_available()
    )


def parse_args():
    parser = argparse.ArgumentParser(description='Train FaceRestore partial-convolution model.')
    parser.add_argument('--train-dir', required=True, help='Directory with square face training images.')
    parser.add_argument('--val-dir', help='Optional validation image directory.')
    parser.add_argument('--masks-dir', required=True, help='Quick Draw ndjson mask source directory.')
    parser.add_argument('--output-dir', default='checkpoints/facerestore', help='Checkpoint output directory.')
    parser.add_argument('--log-dir', default='logs', help='TensorBoard log directory.')
    parser.add_argument('--epochs', type=int, default=100)
    parser.add_argument('--batch-size', type=int, default=16)
    parser.add_argument('--num-workers', type=int, default=4)
    parser.add_argument('--lr', type=float, default=2e-4)
    parser.add_argument('--fine-tune-epoch', type=int, default=50)
    parser.add_argument('--fine-tune-gamma', type=float, default=0.1)
    parser.add_argument('--device', default='cuda' if torch.cuda.is_available() else 'cpu')
    parser.add_argument('--resume')
    parser.add_argument('--start-epoch', type=int, default=0,
                        help='Override starting epoch (use when transitioning from old checkpoints)')
    return parser.parse_args()


def main():
    args = parse_args()
    output_dir = Path(args.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    n_gpus = torch.cuda.device_count()
    device = torch.device(args.device)

    # === TensorBoard (Kaggle-native) ===
    # Logs go to /kaggle/working/logs/ -- viewable via Output tab -> TensorBoard
    writer = SummaryWriter(log_dir=args.log_dir)
    print(f"TensorBoard logging to: {args.log_dir}")
    print("  View in Kaggle: Output tab -> click TensorBoard button")

    # === W&B (optional cloud dashboard) ===
    if HAS_WANDB:
        wandb.init(
            project="facerestore-training",
            config={
                "train_dir": args.train_dir,
                "batch_size": args.batch_size,
                "epochs": args.epochs,
                "lr": args.lr,
                "device": args.device,
                "n_gpus": n_gpus,
                "resume": args.resume or "scratch",
            },
            resume="allow",
        )
        print("W&B cloud logging: ACTIVE (logs at https://wandb.ai)")
    else:
        print("W&B cloud logging: DISABLED")

    print(f"\nTraining config:")
    print(f"  Images:     {args.train_dir}")
    print(f"  Masks:      {args.masks_dir}")
    print(f"  Device:     {args.device}")
    print(f"  GPUs:       {n_gpus}")
    print(f"  Batch size: {args.batch_size} ({args.batch_size // max(n_gpus, 1)} per GPU)")
    print(f"  Workers:    {args.num_workers}")
    print(f"  Epochs:     {args.epochs}")
    print(f"  LR:         {args.lr}")
    print(f"  Resume:     {args.resume or 'None (from scratch)'}")
    print()

    train_loader = make_loader(
        args.train_dir, args.masks_dir, args.batch_size, args.num_workers, val=False
    )
    val_loader = None
    if args.val_dir:
        val_loader = make_loader(
            args.val_dir, args.masks_dir, args.batch_size, args.num_workers, val=True
        )

    model = InpaintNet(bn=True).to(device)
    start_epoch = 1

    if args.resume:
        raw = torch.load(args.resume, map_location=device, weights_only=False)

        if isinstance(raw, dict) and 'model' in raw:
            model_state = raw['model']
            start_epoch = raw.get('epoch', 0) + 1
            print(f"Resumed from {args.resume} -> continuing at epoch {start_epoch}")
        else:
            model_state = raw
            print(f"Resumed from {args.resume} (legacy weights)")

        if args.start_epoch > 0:
            start_epoch = args.start_epoch
            print(f"Manual override: starting at epoch {start_epoch}")

        for key in list(model_state.keys()):
            new_key = key.replace('module.', '')
            model_state[new_key] = model_state.pop(key)
        model.load_state_dict(model_state, strict=False)

    # === Multi-GPU: Wrap with DataParallel if 2+ GPUs ===
    if n_gpus >= 2:
        model = nn.DataParallel(model)
        print(f"DataParallel enabled: {n_gpus} GPUs")
        print(f"  Batch {args.batch_size} split -> {args.batch_size // n_gpus} per GPU")
    else:
        print(f"Single GPU mode ({n_gpus} GPU)")

    criterion = InpaintLoss().to(device)
    optimizer = torch.optim.Adam(
        filter(lambda parameter: parameter.requires_grad, model.parameters()),
        lr=args.lr
    )
    scheduler = torch.optim.lr_scheduler.MultiStepLR(
        optimizer,
        milestones=[args.fine_tune_epoch],
        gamma=args.fine_tune_gamma
    )

    if args.resume and isinstance(raw, dict) and 'optimizer' in raw:
        optimizer.load_state_dict(raw['optimizer'])
        if 'scheduler' in raw:
            scheduler.load_state_dict(raw['scheduler'])
        print("Restored optimizer & scheduler state")

    best_val = None
    end_epoch = start_epoch + args.epochs
    for epoch in range(start_epoch, end_epoch):
        train_loss = run_epoch(model, criterion, train_loader, device, optimizer, epoch_num=epoch)
        val_loss = None
        if val_loader is not None:
            with torch.no_grad():
                val_loss = run_epoch(model, criterion, val_loader, device, epoch_num=epoch)

        scheduler.step()

        # Save epoch checkpoint (clean state_dict, no module. prefix)
        state_dict = get_model_state_dict(model)
        checkpoint = output_dir / 'epoch_{:03d}.pth'.format(epoch)
        torch.save(state_dict, checkpoint)
        if val_loss is not None and (best_val is None or val_loss < best_val):
            best_val = val_loss
            torch.save(state_dict, output_dir / 'best.pth')

        # Save last.pth with full training state
        torch.save({
            'model': state_dict,
            'epoch': epoch,
            'optimizer': optimizer.state_dict(),
            'scheduler': scheduler.state_dict(),
        }, output_dir / 'last.pth')

        current_lr = optimizer.param_groups[0]['lr']
        message = 'epoch={} train_loss={:.6f} lr={:.6g}'.format(epoch, train_loss, current_lr)
        if val_loss is not None:
            message += ' val_loss={:.6f}'.format(val_loss)
        print(message, flush=True)

        # === Log to Kaggle TensorBoard ===
        writer.add_scalar('Loss/train', train_loss, epoch)
        if val_loss is not None:
            writer.add_scalar('Loss/val', val_loss, epoch)
        writer.add_scalar('LR', current_lr, epoch)
        writer.flush()

        # === Log to W&B ===
        if HAS_WANDB:
            log_data = {"train_loss": train_loss, "lr": current_lr, "epoch": epoch}
            if val_loss is not None:
                log_data["val_loss"] = val_loss
            wandb.log(log_data)

    writer.close()
    if HAS_WANDB:
        wandb.finish()
        print("\nLogging complete.")
        print("  TensorBoard: Output tab -> TensorBoard button")
        print("  W&B: https://wandb.ai")


if __name__ == '__main__':
    main()
'''

# Write the modified train.py
train_py_path = os.path.join(PROJECT_DIR, 'train.py')
with open(train_py_path, 'w') as f:
    f.write(TRAIN_PY)

# Also copy into the inpaint subpackage
inpaint_train_py = os.path.join(PROJECT_DIR, 'inpaint', 'train.py')
shutil.copy2(train_py_path, inpaint_train_py)

print(f'Written modified train.py to:')
print(f'  {train_py_path}')
print(f'  {inpaint_train_py}')
print()
print('Changes from Colab version:')
print('  + nn.DataParallel auto-wrap for multi-GPU')
print('  + get_model_state_dict() - saves clean checkpoints (no module. prefix)')
print('  + Kaggle TensorBoard logging (built-in, zero setup)')
print('  + W&B logging (optional, for cloud dashboard)')
print('  + --log-dir argument for TensorBoard output')
print('  + Default workers: 2 -> 4')

In [ ]:
# Cell 5: Run Training
# Auto-detects whether to resume from last.pth or start from base model

import os
os.chdir(os.path.join(PROJECT_DIR, 'inpaint'))

# Priority: last.pth (full training state) > base model.state_dict
last_ckpt = os.path.join(CKPT_DIR, 'last.pth')
base_model = os.path.join(WORK_DIR, 'model.state_dict')

if os.path.exists(last_ckpt):
    resume_from = last_ckpt
    print(f'Resuming from last checkpoint: {last_ckpt}')
elif os.path.exists(base_model):
    resume_from = base_model
    print(f'No checkpoint found. Starting from base model: {base_model}')
else:
    resume_from = None
    print('WARNING: No checkpoint or base model found! Training from scratch.')

# === Training Arguments ===
BATCH_SIZE = 32    # 16 per GPU with 2x T4 (DataParallel splits automatically)
NUM_WORKERS = 4    # Kaggle gives more CPU cores than Colab
EPOCHS = 15        # ~2 hrs/epoch * 15 = 30 hrs (full weekly quota)
LR = 2e-5          # Same LR as your Colab runs

!python train.py \
  --train-dir {DATASET_DIR}/faces \
  --masks-dir {DATASET_DIR}/masks \
  --resume {resume_from} \
  --output-dir {CKPT_DIR} \
  --log-dir {TB_LOG_DIR} \
  --device cuda:0 \
  --batch-size {BATCH_SIZE} \
  --num-workers {NUM_WORKERS} \
  --epochs {EPOCHS} \
  --lr {LR}

In [ ]:
# Cell 6: Check Saved Checkpoints + TensorBoard Logs
import os

print('='*60)
print('CHECKPOINTS')
print('='*60)
if os.path.exists(CKPT_DIR):
    files = sorted(os.listdir(CKPT_DIR))
    print(f'\nSaved checkpoints ({len(files)} files):')
    total_size = 0
    for f in files:
        size_mb = os.path.getsize(os.path.join(CKPT_DIR, f)) / (1024*1024)
        total_size += size_mb
        print(f'  {f} ({size_mb:.1f} MB)')
    print(f'\nTotal size: {total_size:.1f} MB')
else:
    print('No checkpoint directory found.')

# Show last.pth epoch info
import torch
last_path = os.path.join(CKPT_DIR, 'last.pth')
if os.path.exists(last_path):
    data = torch.load(last_path, map_location='cpu', weights_only=False)
    if isinstance(data, dict) and 'epoch' in data:
        print(f'\nLast checkpoint epoch: {data["epoch"]}')
        print(f'Next training will start from epoch: {data["epoch"] + 1}')

print('\n' + '='*60)
print('TENSORBOARD LOGS')
print('='*60)
if os.path.exists(TB_LOG_DIR):
    for item in os.listdir(TB_LOG_DIR):
        print(f'  {item}')
    print(f'\nView these in Kaggle: Output tab -> TensorBoard button')
else:
    print('No TensorBoard logs found yet.')

In [ ]:
# Cell 7: Package Checkpoints for Download / Re-upload
#
# Run this after training to create a downloadable archive.
# For the NEXT Kaggle session, upload this zip as a new version of your
# 'facerestore-checkpoints' dataset so training auto-resumes.
#
# Only last.pth is strictly needed for resuming, but we package
# everything so you also keep the per-epoch and best checkpoints.

import shutil, os

archive_name = 'facerestore_checkpoints'
archive_path = os.path.join(WORK_DIR, archive_name)

if os.path.exists(CKPT_DIR) and len(os.listdir(CKPT_DIR)) > 0:
    shutil.make_archive(archive_path, 'zip', CKPT_DIR)
    size_mb = os.path.getsize(archive_path + '.zip') / (1024*1024)
    print(f'Created: {archive_path}.zip ({size_mb:.1f} MB)')
    print(f'\nTo resume training next session:')
    print(f'  1. Download {archive_name}.zip from the Output tab')
    print(f'  2. Go to your facerestore-checkpoints dataset on Kaggle')
    print(f'  3. Click "New Version" and upload the zip')
    print(f'  4. In your next notebook run, it will auto-resume from last.pth')
    print(f'\nNote: last.pth is all you need to resume. The other epoch_*.pth')
    print(f'files are included for backup but are not required for resuming.')
else:
    print('No checkpoints to package.')

## Resuming Training (Next Session)

Unlike Colab, Kaggle doesn't have persistent Drive storage between sessions. Here's the workflow:

1. **After training completes (or session ends):**
   - Run Cell 7 to package checkpoints into a zip
   - Download the zip from the **Output tab** on the right

2. **Before the next session:**
   - Go to your **facerestore-checkpoints** dataset on Kaggle
   - Click **New Version** -> Upload the zip (or just `last.pth`)
   - This updates the dataset that the notebook reads from

3. **In the next session:**
   - Run Cells 1-4 (setup + write train.py)
   - Run Cell 5 -- it **automatically** detects `last.pth` and resumes

No manual epoch editing needed. The `last.pth` file stores the epoch number, optimizer state, and scheduler state, so training picks up exactly where it left off.

### Do I need all the epoch_00X.pth files?

**No.** Only `last.pth` is required for resuming. The individual epoch files are just raw model weights without training state. You can upload just `last.pth` to save time.

### Quick Estimate: How Many Sessions to 50 Epochs?

| Current | Per Session | Sessions Needed |
|---------|-------------|----------------|
| 5 epochs | +15 epochs | ~3 sessions to reach 50 |

At ~30 hrs/week and ~2 hrs/epoch with dual T4, you get roughly 15 epochs per week. Starting from epoch 5, you'll hit epoch 50 in about 3 weekly sessions.

### Viewing Metrics

- **Kaggle TensorBoard**: Output tab -> click the TensorBoard button. Shows loss curves, learning rate, etc. No setup required -- it just works.
- **W&B**: If you set up your API key as a Kaggle Secret, metrics also appear at https://wandb.ai with full run history across sessions.